## Pre match prediction

In [64]:
import pandas as pd
import numpy as np
import joblib

In [65]:
# Load the datasets
matches = pd.read_csv('..\data\matches_new.csv')
matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,year
0,829705,2015,Kolkata,2015-04-08,League,M Morkel,Eden Gardens,KKR,MI,KKR,...,KKR,wickets,7.0,169.0,20.0,N,NaN,S Ravi,C Shamshuddin,2015
1,829707,2015,Chennai,2015-04-09,League,A Nehra,MA Chidambaram Stadium,CSK,DC,DC,...,CSK,runs,1.0,151.0,20.0,N,NaN,RK Illingworth,VA Kulkarni,2015
2,829709,2015,Pune,2015-04-10,League,JP Faulkner,Maharashtra Cricket Association Stadium,PBKS,RR,PBKS,...,RR,runs,26.0,163.0,20.0,N,NaN,SD Fry,CB Gaffaney,2015
3,829711,2015,Chennai,2015-04-11,League,BB McCullum,MA Chidambaram Stadium,CSK,SRH,CSK,...,CSK,runs,45.0,210.0,20.0,N,NaN,RK Illingworth,VA Kulkarni,2015
4,829713,2015,Kolkata,2015-04-11,League,CH Gayle,Eden Gardens,KKR,RCB,RCB,...,RCB,wickets,3.0,178.0,20.0,N,NaN,S Ravi,C Shamshuddin,2015


In [66]:
print("Match INFO\n")
matches.info()
print("Match Details\n")
matches.describe()
matches.columns

Match INFO

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 581 entries, 0 to 580
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               581 non-null    int64  
 1   season           581 non-null    object 
 2   city             543 non-null    object 
 3   date             581 non-null    object 
 4   match_type       581 non-null    object 
 5   player_of_match  577 non-null    object 
 6   venue            581 non-null    object 
 7   team1            581 non-null    object 
 8   team2            581 non-null    object 
 9   toss_winner      581 non-null    object 
 10  toss_decision    581 non-null    object 
 11  winner           577 non-null    object 
 12  result           581 non-null    object 
 13  result_margin    569 non-null    float64
 14  target_runs      579 non-null    float64
 15  target_overs     579 non-null    float64
 16  super_over       581 non-null    object 
 17  meth

Index(['id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2', 'year'],
      dtype='object')

In [67]:
matches = pd.read_csv("../data/matches_new.csv")

needCol = [
    'season','city','date',
    'team1','team2',
    'toss_winner','toss_decision',
    'winner','venue'
]

matches = matches[needCol]


matches['city'].fillna("Unknown", inplace=True)

C:\Users\hazra\AppData\Local\Temp\ipykernel_11676\3079068023.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matches['city'].fillna("Unknown", inplace=True)


In [68]:
matches.head()

,season,city,date,team1,team2,toss_winner,toss_decision,winner,venue
0,2015,Kolkata,2015-04-08,KKR,MI,KKR,field,KKR,Eden Gardens
1,2015,Chennai,2015-04-09,CSK,DC,DC,field,CSK,MA Chidambaram Stadium
2,2015,Pune,2015-04-10,PBKS,RR,PBKS,field,RR,Maharashtra Cricket Association Stadium
3,2015,Chennai,2015-04-11,CSK,SRH,CSK,bat,CSK,MA Chidambaram Stadium
4,2015,Kolkata,2015-04-11,KKR,RCB,RCB,field,RCB,Eden Gardens


In [69]:
team_form = {}

team1_last5 = []
team2_last5 = []

for i in matches.index:

    t1 = matches.loc[i,'team1']
    t2 = matches.loc[i,'team2']
    winner = matches.loc[i,'winner']

    last5_t1 = team_form.get(t1, [])[-5:]
    last5_t2 = team_form.get(t2, [])[-5:]

    team1_last5.append(sum(last5_t1)/5 if last5_t1 else 0.5)
    team2_last5.append(sum(last5_t2)/5 if last5_t2 else 0.5)

    team_form.setdefault(t1,[])
    team_form.setdefault(t2,[])

    if winner == t1:
        team_form[t1].append(1)
        team_form[t2].append(0)
    else:
        team_form[t1].append(0)
        team_form[t2].append(1)

matches['team1_form'] = team1_last5
matches['team2_form'] = team2_last5

In [70]:
h2h = {}

t1_h2h=[]
t2_h2h=[]

for i in matches.index:

    t1 = matches.loc[i,'team1']
    t2 = matches.loc[i,'team2']
    winner = matches.loc[i,'winner']

    pair=(t1,t2)
    rev=(t2,t1)

    t1wins=h2h.get(pair,0)
    t2wins=h2h.get(rev,0)

    t1_h2h.append(t1wins)
    t2_h2h.append(t2wins)

    if winner==t1:
        h2h[pair]=t1wins+1
    else:
        h2h[rev]=t2wins+1

matches['team1_h2h']=t1_h2h
matches['team2_h2h']=t2_h2h

In [71]:
venue_stats={}

t1_venue=[]
t2_venue=[]

for i in matches.index:

    t1=matches.loc[i,'team1']
    t2=matches.loc[i,'team2']
    venue=matches.loc[i,'venue']
    winner=matches.loc[i,'winner']

    k1=(t1,venue)
    k2=(t2,venue)

    w1,m1=venue_stats.get(k1,(0,0))
    w2,m2=venue_stats.get(k2,(0,0))

    t1_venue.append(w1/m1 if m1 else 0.5)
    t2_venue.append(w2/m2 if m2 else 0.5)

    if winner==t1:
        venue_stats[k1]=(w1+1,m1+1)
        venue_stats[k2]=(w2,m2+1)
    else:
        venue_stats[k1]=(w1,m1+1)
        venue_stats[k2]=(w2+1,m2+1)

matches['team1_venue_wr']=t1_venue
matches['team2_venue_wr']=t2_venue

In [72]:
team_stats={}

t1_wr=[]
t2_wr=[]

for i in matches.index:

    t1=matches.loc[i,'team1']
    t2=matches.loc[i,'team2']
    winner=matches.loc[i,'winner']

    w1,m1=team_stats.get(t1,(0,0))
    w2,m2=team_stats.get(t2,(0,0))

    t1_wr.append(w1/m1 if m1 else 0.5)
    t2_wr.append(w2/m2 if m2 else 0.5)

    if winner==t1:
        team_stats[t1]=(w1+1,m1+1)
        team_stats[t2]=(w2,m2+1)
    else:
        team_stats[t1]=(w1,m1+1)
        team_stats[t2]=(w2+1,m2+1)

matches['team1_wr']=t1_wr
matches['team2_wr']=t2_wr

In [73]:
matches['toss_win_team1'] = (
    matches['toss_winner']==matches['team1']
).astype(int)

matches['team1_batting_first']=(
    ((matches['toss_winner']==matches['team1']) &
     (matches['toss_decision']=='bat')) |
    ((matches['toss_winner']==matches['team2']) &
     (matches['toss_decision']=='field'))
).astype(int)

In [74]:
matches['win_rate_diff']=matches['team1_wr']-matches['team2_wr']
matches['form_diff']=matches['team1_form']-matches['team2_form']
matches['h2h_diff']=matches['team1_h2h']-matches['team2_h2h']
matches['venue_diff']=matches['team1_venue_wr']-matches['team2_venue_wr']

In [75]:
matches.columns

Index(['season', 'city', 'date', 'team1', 'team2', 'toss_winner',
       'toss_decision', 'winner', 'venue', 'team1_form', 'team2_form',
       'team1_h2h', 'team2_h2h', 'team1_venue_wr', 'team2_venue_wr',
       'team1_wr', 'team2_wr', 'toss_win_team1', 'team1_batting_first',
       'win_rate_diff', 'form_diff', 'h2h_diff', 'venue_diff'],
      dtype='object')

In [76]:
X = matches[[
'team1','team2',
'toss_winner','toss_decision',
'venue',
'win_rate_diff','form_diff',
'h2h_diff','venue_diff',
'toss_win_team1','team1_batting_first'
]]

y = (matches['winner']==matches['team1']).astype(int)

In [77]:
X = pd.get_dummies(
    X,
    columns=['team1','team2','toss_winner','toss_decision','venue'],
    drop_first=True
)

In [78]:
# train model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Split the datasets
split=int(len(X)*0.8)

X_train=X.iloc[:split]
X_test=X.iloc[split:]

y_train=y.iloc[:split]
y_test=y.iloc[split:]


prematch_prediction = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

prematch_prediction.fit(X_train, y_train)

# evaluation
y_pred = prematch_prediction.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.47863247863247865


In [79]:
joblib.dump(prematch_prediction,"../models/prematch_model.pkl")
joblib.dump(X.columns.tolist(),"../models/features.pkl")
# joblib.dump(encoders,"models/encoders.pkl")
joblib.dump(h2h,"../models/h2h.pkl")
joblib.dump(venue_stats,"../models/venue_stats.pkl")
joblib.dump(team_stats,"../models/team_stats.pkl")

['../models/team_stats.pkl']